<a href="https://colab.research.google.com/github/phamhoquephuong/PHAMHOQUEPHUONG-BTVN2/blob/main/MenuCheckoutFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob
import zipfile
import shutil
import re

PROJECT_DIR = "/content/drive/MyDrive/canteen_project"
ZIP_DIR = os.path.join(PROJECT_DIR, "zips")

RAW_DIR = "/content/canteen_raw"
DATASET_DIR = "/content/canteen_dataset"

if os.path.exists(RAW_DIR):
    shutil.rmtree(RAW_DIR)

if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

print("ZIP_DIR:", ZIP_DIR)
print("RAW_DIR:", RAW_DIR)
print("DATASET_DIR:", DATASET_DIR)

In [ ]:
from PIL import Image

valid_exts = [".jpg", ".jpeg", ".png", ".webp"]

def clean_class_name(zip_filename):
    name = os.path.basename(zip_filename)
    name = name.replace(".zip", "")
    name = re.sub(r"-2026.*", "", name)
    name = name.lower()
    return name

zip_files = sorted(glob.glob(os.path.join(ZIP_DIR, "*.zip")))

print("Tổng số file zip:", len(zip_files))

for zip_path in zip_files:
    class_name = clean_class_name(zip_path)
    class_dir = os.path.join(RAW_DIR, class_name)
    os.makedirs(class_dir, exist_ok=True)

    temp_dir = "/content/temp_extract"

    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    os.makedirs(temp_dir, exist_ok=True)

    print("Đang giải nén:", os.path.basename(zip_path), "->", class_name)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(temp_dir)

    image_paths = []

    for root, dirs, files in os.walk(temp_dir):
        for file in files:
            ext = os.path.splitext(file)[1].lower()
            if ext in valid_exts:
                image_paths.append(os.path.join(root, file))

    for i, img_path in enumerate(image_paths):
        ext = os.path.splitext(img_path)[1].lower()
        new_name = f"{class_name}_{i:05d}{ext}"
        new_path = os.path.join(class_dir, new_name)
        shutil.copy(img_path, new_path)

    print("Số ảnh copy:", len(image_paths))

print("Giải nén xong toàn bộ.")

In [ ]:
for cls in sorted(os.listdir(RAW_DIR)):
    cls_path = os.path.join(RAW_DIR, cls)

    if os.path.isdir(cls_path):
        count = len(glob.glob(cls_path + "/*"))
        print(f"{cls}: {count} ảnh")

In [ ]:
import hashlib
from PIL import Image

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

removed_bad = 0
removed_dup = 0

for cls in os.listdir(RAW_DIR):
    cls_path = os.path.join(RAW_DIR, cls)

    if not os.path.isdir(cls_path):
        continue

    seen_hashes = set()

    for file in os.listdir(cls_path):
        file_path = os.path.join(cls_path, file)

        try:
            img = Image.open(file_path)
            img.verify()
        except:
            os.remove(file_path)
            removed_bad += 1
            continue

        h = file_hash(file_path)

        if h in seen_hashes:
            os.remove(file_path)
            removed_dup += 1
        else:
            seen_hashes.add(h)

print("Ảnh lỗi đã xóa:", removed_bad)
print("Ảnh trùng đã xóa:", removed_dup)

In [ ]:
print("Số ảnh sau khi lọc:")

for cls in sorted(os.listdir(RAW_DIR)):
    cls_path = os.path.join(RAW_DIR, cls)

    if os.path.isdir(cls_path):
        count = len(glob.glob(cls_path + "/*"))
        print(f"{cls}: {count} ảnh")

In [ ]:
!pip install split-folders

In [ ]:
import splitfolders

splitfolders.ratio(
    RAW_DIR,
    output=DATASET_DIR,
    seed=42,
    ratio=(0.7, 0.15, 0.15)
)

print("Đã chia dataset xong.")

In [ ]:
for split in ["train", "val", "test"]:
    print("\n" + split.upper())

    split_path = os.path.join(DATASET_DIR, split)

    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)

        if os.path.isdir(cls_path):
            count = len(glob.glob(cls_path + "/*"))
            print(f"{cls}: {count} ảnh")

In [ ]:
import tensorflow as tf

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "val")
TEST_DIR = os.path.join(DATASET_DIR, "test")

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Class names:", class_names)
print("Number of classes:", num_classes)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
from tensorflow.keras import layers, models

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),

    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

checkpoint_path = "/content/best_canteen_11classes.keras"

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=callbacks
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Training and Validation Accuracy")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training and Validation Loss")
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)

print("Test accuracy:", test_acc)
print("Test loss:", test_loss)

In [ ]:
PRICE_TABLE = {
    "com_trang": 10000,
    "dau_hu_sot_ca": 25000,
    "ca_hu_kho": 30000,
    "thit_kho_trung": 30000,
    "thit_kho": 25000,
    "canh_chua_co_ca": 25000,
    "canh_chua_khong_ca": 10000,
    "suon_nuong": 30000,
    "canh_rau": 7000,
    "rau_xao": 10000,
    "trung_chien": 25000
}

In [ ]:
for cls in class_names:
    if cls not in PRICE_TABLE:
        print("Chưa có giá:", cls)

In [ ]:
import random
import numpy as np

test_images = []

for cls in class_names:
    cls_path = os.path.join(TEST_DIR, cls)
    test_images.extend(glob.glob(cls_path + "/*"))

sample_image = random.choice(test_images)

print("Ảnh test:", sample_image)

In [ ]:
def predict_one_image(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    pred_idx = np.argmax(preds)

    dish_name = class_names[pred_idx]
    confidence = preds[pred_idx]
    price = PRICE_TABLE.get(dish_name, 0)

    plt.imshow(tf.keras.utils.load_img(image_path))
    plt.axis("off")
    plt.title(f"{dish_name} - {confidence:.2%}")
    plt.show()

    print("Món dự đoán:", dish_name)
    print("Độ tin cậy:", f"{confidence:.2%}")
    print("Giá:", f"{price:,} VND")

    return dish_name, confidence, price

In [ ]:
save_path = "/content/drive/MyDrive/canteen_project/canteen_cnn_11classes.keras"

model.save(save_path)

print("Đã lưu model:", save_path)

## Test 1 ảnh khay cơm tổng hợp: crop từng món → dự đoán → tính bill

Phần này dùng sau khi model CNN đã train xong.  
Luồng xử lý:

`ảnh khay cơm tổng hợp → nhập tọa độ từng món → crop từng món → CNN dự đoán → tra bảng giá → tính tổng bill`


In [ ]:
# Upload 1 ảnh khay cơm tổng hợp từ máy lên Colab

from google.colab import files
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

uploaded = files.upload()

tray_image_path = list(uploaded.keys())[0]
print("Ảnh khay cơm đã upload:", tray_image_path)

In [ ]:
# Hiển thị ảnh khay cơm để xem kích thước và lấy tọa độ crop

def show_tray_image(image_path):
    img_bgr = cv2.imread(image_path)

    if img_bgr is None:
        raise ValueError("Không đọc được ảnh. Kiểm tra lại đường dẫn ảnh.")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    height, width = img_rgb.shape[:2]

    print(f"Kích thước ảnh: width={width}, height={height}")
    print("Khi nhập boxes, dùng format: (x, y, width, height)")

    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.axis("on")
    plt.grid()
    plt.title("Ảnh khay cơm tổng hợp")
    plt.show()

show_tray_image(tray_image_path)

### Nhập tọa độ crop từng món

Sửa danh sách `boxes` bên dưới theo ảnh khay cơm của bạn.  
Mỗi dòng là một món ăn, theo format:

`(x, y, width, height)`

Ví dụ: `(40, 50, 250, 220)` nghĩa là bắt đầu từ điểm `x=40`, `y=50`, cắt vùng rộng `250`, cao `220`.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(img_path)

fig = plt.figure(figsize=(12,8))
plt.imshow(img)

print("Click các góc trên ảnh")

pts = plt.ginput(2)
plt.show()

print(pts)

In [ ]:
import cv2

img = cv2.imread(tray_image_path)

# Kéo chuột chọn nhiều vùng
# rois = cv2.selectROIs(
#     "Chon mon an",
#     img,
#     showCrosshair=True,
#     fromCenter=False
# )

# print(rois)

# cv2.destroyAllWindows()

print("Vui lòng định nghĩa danh sách 'boxes' thủ công trong ô tiếp theo thay vì sử dụng selectROIs.")

In [ ]:
# Xem trước các khung crop trên ảnh gốc

def preview_boxes_on_tray(image_path, boxes):
    img_bgr = cv2.imread(image_path)

    if img_bgr is None:
        raise ValueError("Không đọc được ảnh. Kiểm tra lại đường dẫn ảnh.")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    preview = img_rgb.copy()

    for i, (x, y, w, h) in enumerate(boxes, start=1):
        cv2.rectangle(preview, (x, y), (x + w, y + h), (255, 0, 0), 4)
        cv2.putText(
            preview,
            f"Dish {i}",
            (x, max(y - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 0, 0),
            3
        )

    plt.figure(figsize=(10, 8))
    plt.imshow(preview)
    plt.axis("off")
    plt.title("Preview vùng crop từng món")
    plt.show()

preview_boxes_on_tray(tray_image_path, boxes)

In [ ]:
# Crop từng món và lưu thành ảnh riêng

def crop_dishes_from_tray(image_path, boxes, output_dir="/content/cropped_dishes_from_tray"):
    os.makedirs(output_dir, exist_ok=True)

    # Xóa ảnh crop cũ để tránh bị lẫn với lần chạy trước
    for file in os.listdir(output_dir):
        file_path = os.path.join(output_dir, file)
        if os.path.isfile(file_path):
            os.remove(file_path)

    img_bgr = cv2.imread(image_path)

    if img_bgr is None:
        raise ValueError("Không đọc được ảnh. Kiểm tra lại đường dẫn ảnh.")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_h, img_w = img_rgb.shape[:2]

    cropped_paths = []

    for i, (x, y, w, h) in enumerate(boxes, start=1):
        # Giới hạn tọa độ để không vượt ra ngoài ảnh
        x1 = max(0, x)
        y1 = max(0, y)
        x2 = min(img_w, x + w)
        y2 = min(img_h, y + h)

        crop = img_rgb[y1:y2, x1:x2]

        if crop.size == 0:
            print(f"Bỏ qua món {i}: vùng crop rỗng.")
            continue

        save_path = os.path.join(output_dir, f"dish_{i}.jpg")
        cv2.imwrite(save_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        cropped_paths.append(save_path)

    return cropped_paths

cropped_paths = crop_dishes_from_tray(tray_image_path, boxes)

print("Đã crop xong các món:")
for path in cropped_paths:
    print(path)

In [ ]:
# Hiển thị các ảnh món đã crop

def show_cropped_dishes(cropped_paths):
    if len(cropped_paths) == 0:
        print("Chưa có ảnh crop nào.")
        return

    plt.figure(figsize=(4 * len(cropped_paths), 4))

    for i, path in enumerate(cropped_paths):
        img_bgr = cv2.imread(path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        plt.subplot(1, len(cropped_paths), i + 1)
        plt.imshow(img_rgb)
        plt.title(f"Dish {i + 1}")
        plt.axis("off")

    plt.show()

show_cropped_dishes(cropped_paths)

In [ ]:
# Dự đoán từng ảnh crop, tra giá từng món và tính tổng bill

def predict_one_dish_silent(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    pred_idx = int(np.argmax(preds))

    dish_name = class_names[pred_idx]
    confidence = float(preds[pred_idx])
    price = int(PRICE_TABLE.get(dish_name, 0))

    return dish_name, confidence, price


def calculate_bill_from_cropped_images(cropped_paths):
    bill_rows = []
    total = 0

    print("===== CANTEEN BILL =====")

    for i, image_path in enumerate(cropped_paths, start=1):
        dish_name, confidence, price = predict_one_dish_silent(image_path)
        total += price

        bill_rows.append({
            "STT": i,
            "Image": image_path,
            "Dish": dish_name,
            "Confidence": f"{confidence:.2%}",
            "Price (VND)": price
        })

        print(f"Món {i}: {dish_name}")
        print(f"Độ tin cậy: {confidence:.2%}")
        print(f"Giá: {price:,} VND")
        print("------------------------")

    print(f"TỔNG BILL: {total:,} VND")

    bill_df = pd.DataFrame(bill_rows)
    return bill_df, total


bill_df, total_bill = calculate_bill_from_cropped_images(cropped_paths)

display(bill_df)
print("Tổng bill:", f"{total_bill:,} VND")

In [ ]:
# Hiển thị ảnh crop kèm kết quả dự đoán và giá tiền

def show_bill_result_grid(bill_df):
    if bill_df.empty:
        print("Không có kết quả bill để hiển thị.")
        return

    n = len(bill_df)
    plt.figure(figsize=(4 * n, 4))

    for idx, row in bill_df.iterrows():
        img_bgr = cv2.imread(row["Image"])
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        title = f'{row["Dish"]}\n{row["Price (VND)"]:,} VND\n{row["Confidence"]}'

        plt.subplot(1, n, idx + 1)
        plt.imshow(img_rgb)
        plt.title(title)
        plt.axis("off")

    plt.show()

    print("TỔNG BILL:", f"{total_bill:,} VND")

show_bill_result_grid(bill_df)

In [ ]:
# Lưu hóa đơn ra file CSV nếu cần

bill_csv_path = "/content/canteen_bill_result.csv"
bill_df.to_csv(bill_csv_path, index=False, encoding="utf-8-sig")

print("Đã lưu hóa đơn:", bill_csv_path)

**Phần dưới là chụp ảnh trực tiếp**

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='tray_webcam.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const button = document.createElement('button');
            button.textContent = 'Chụp ảnh khay cơm';
            div.appendChild(button);

            const video = document.createElement('video');
            video.style.display = 'block';
            video.width = 800;

            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);

            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            await new Promise((resolve) => button.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;

            canvas.getContext('2d').drawImage(video, 0, 0);

            stream.getVideoTracks()[0].stop();
            div.remove();

            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')

    display(js)
    data = eval_js('takePhoto({})'.format(quality))

    binary = b64decode(data.split(',')[1])

    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

In [ ]:
tray_image_path = take_photo()
print("Ảnh đã chụp từ webcam:", tray_image_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread(tray_image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.axis("on")
plt.title("Ảnh khay cơm từ webcam")
plt.show()

In [ ]:
boxes = [
    (10, 10, 220, 220), # Món 1: Fried eggs
    (230, 10, 200, 220), # Món 2: Braised pork with egg
    (440, 10, 190, 220), # Món 3: Grilled meat
    (10, 250, 220, 220), # Món 4: White rice
    (250, 250, 380, 220)  # Món 5: Soup
]

cropped_paths = crop_dishes_from_tray(tray_image_path, boxes)

bill_results, total = calculate_bill_from_cropped_images(cropped_paths)

In [ ]:
!pip install qrcode[pil]
import ipywidgets as widgets
from IPython.display import display, clear_output
import qrcode
from PIL import Image

# 1. Dữ liệu món ăn từ yêu cầu
menu = {
    "Cơm trắng": 10000,
    "Đậu hũ sốt cà": 25000,
    "Cá hú kho": 30000,
    "Thịt kho trứng": 30000,
    "Thịt kho": 25000,
    "Canh chua có cá": 25000,
    "Canh chua không cá": 10000,
    "Sườn nướng": 30000,
    "Canh rau": 7000,
    "Rau xào": 10000,
    "Trứng chiên": 25000
}

# 2. Tạo các widget giao diện
output = widgets.Output()
checkboxes = {item: widgets.Checkbox(value=False, description=f"{item} ({price}đ)") for item, price in menu.items()}
button = widgets.Button(description="Tính tiền & Tạo QR", button_style='success')

def generate_qr(total_amount):
    # Dữ liệu mã QR có thể chứa thông tin thanh toán (ví dụ: link ngân hàng hoặc thông tin bill)
    data = f"Total Payment: {total_amount} VND"
    qr = qrcode.make(data)
    qr.save("payment_qr.png")
    return "payment_qr.png"

def on_button_clicked(b):
    total = 0
    selected_items = []
    for item, box in checkboxes.items():
        if box.value:
            total += menu[item]
            selected_items.append(item)

    with output:
        clear_output()
        print(f"Các món đã chọn: {', '.join(selected_items)}")
        print(f"--- Tổng hóa đơn: {total} VNĐ ---")

        if total > 0:
            qr_path = generate_qr(total)
            display(Image.open(qr_path))
            print("Mã QR thanh toán đã được tạo.")
        else:
            print("Vui lòng chọn ít nhất một món ăn.")

button.on_click(on_button_clicked)

# 3. Hiển thị giao diện
ui = widgets.VBox([widgets.Label("Chọn món ăn:")] + list(checkboxes.values()) + [button, output])
display(ui)